# Set-up  

In [1]:
from google import genai
from google.genai import types


import time
import os
import pandas as pd
from datetime import datetime

PROMPTS_DIR = './prompts'

def load_prompt(agent_type, model, name, version="v02"):
    """Load a prompt from a .md file in the prompts directory."""
    path = os.path.join(PROMPTS_DIR, agent_type, model, f"{name}_{version}.md")
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().strip()

# Gemini Client Initiation

In [21]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
client

In [22]:
# List all documents across all file search stores
for fs in client.file_search_stores.list():
    print(f"Documents in {fs.display_name} ({fs.name}):")
    docs = client.file_search_stores.documents.list(parent=fs.name)
    for doc in docs:
        print(f"  - {doc.name}: {doc.display_name}")
    print()

Documents in Legal Information (fileSearchStores/legal-information-iha7u4ocs45v):
  - fileSearchStores/legal-information-iha7u4ocs45v/documents/the-civil-and-commercial-co-bxga6ria2fwj: The Civil and Commercial Code_20251111.pdf

Documents in SNS (fileSearchStores/sns-d4miedjtv6mf):
  - fileSearchStores/sns-d4miedjtv6mf/documents/analysisoflegal1pdf-bo1hpblnpapx: Analysis_of_Legal1.pdf

Documents in L&C-Public Company Law Documents (fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5):
  - fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/public-company-lawactpublic-vvd67tmf5qs1: Public Company Law/Act_Public_Company_2535_20260204.pdf

Documents in L&C-General Law Documents (fileSearchStores/lcgeneral-law-documents-8uyfomy1ah0b):
  - fileSearchStores/lcgeneral-law-documents-8uyfomy1ah0b/documents/general-lawactbankruptcy202-s846352obylm: General Law/Act_Bankruptcy_20260204.pdf
  - fileSearchStores/lcgeneral-law-documents-8uyfomy1ah0b/documents/general-lawactbusi

In [23]:
# client.file_search_stores.documents.delete(name="fileSearchStores/lcgeneral-law-documents-8uyfomy1ah0b/documents/public-company-lawactpublic-v0xozj9qf036",config={'force': True})

In [24]:
if len(client.file_search_stores.list()) < 1:
  # File name will be visible in citations
  file_search_store = client.file_search_stores.create(
      config={
        # 'display_name': 'L&C-Public Company Law Documents',
        # 'display_name': 'L&C-General Law Documents',
        
        'display_name': 'L&C-Supreme Court Statements'
      })
  FILE_STORE_NAME = file_search_store.name
  FILE_STORE_NAME

In [25]:
# FILE_STORE_NAME = "fileSearchStores/legal-information-jjbq4kr8yz7z"
# FILE_STORE_NAME = "fileSearchStores/legal-information-iha7u4ocs45v" ## New One

GENERAL_LAW_FILE_STORE = "fileSearchStores/lcgeneral-law-documents-8uyfomy1ah0b"
PUBLIC_COMPANY_LAW_FILE_STORE = "fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5"
SUPREME_COURT_STATEMENT_FILE_STORE = 'fileSearchStores/lcsupreme-court-statements-v0oghu9g8b7f'

In [26]:
# client.file_search_stores.delete(name='fileSearchStores/lcgeneral-law-documents-5ued446cduu7', config={'force':True})

# Add File To File Search Store

In [8]:
def store_file(client, file_name, file_search_store_name, file_path='./documents/KKP/LNC', metadata=[]):
  full_file_path = os.path.join(file_path, file_name)
  print(f"Storing File: {full_file_path}")
  print(metadata)

  operation = client.file_search_stores.upload_to_file_search_store(
    file=full_file_path,
    file_search_store_name=file_search_store_name,
    config={
        'display_name' : file_name,
        'custom_metadata': metadata
    }
  )
  return operation

In [9]:
# # Walk through sub-folders and upload each .docx file
# base_path = './documents/KKP/LNC/Supreme Court Statement'
# for category in sorted(os.listdir(base_path)):
#     category_path = os.path.join(base_path, category)
#     if not os.path.isdir(category_path):
#         continue

#     for fname in sorted(os.listdir(category_path)):
#         if not fname.endswith('.docx'):
#             continue

#         file_rel_path = os.path.join('Supreme Court Statement', category, fname)
#         metadata = [
#             {'key': 'law_level', 'string_value': 'คำพิพากษาศาลฎีกา'},
#             {'key': 'law_type', 'string_value': 'คำพิพากษาศาลฎีกา'},
#             {'key': 'legal_category', 'string_value': category},
#         ]

#         operation = store_file(client, file_rel_path, SUPREME_COURT_STATEMENT_FILE_STORE, metadata=metadata)

#         while not operation.done:
#             time.sleep(5)
#             operation = client.operations.get(operation)

#         print(f"  ✓ Uploaded: {fname}")

# print(f"\nDone! All documents uploaded to {SUPREME_COURT_STATEMENT_FILE_STORE}")

# # Verify
# docs = client.file_search_stores.documents.list(parent=SUPREME_COURT_STATEMENT_FILE_STORE)
# print(f"\nDocuments in store ({len(list(docs))}):")
# for doc in client.file_search_stores.documents.list(parent=SUPREME_COURT_STATEMENT_FILE_STORE):
#     print(f"  - {doc.display_name}")

In [10]:
# # file_list = [
# #   #  'The Civil and Commercial Code_20251111.pdf'
# #   # '20240702_BOT_Policy.pdf',
# #   # 'Analysis_of_Legal1.pdf'

# #   "General Law/Act_Bankruptcy_20260204.pdf",
# #   "General Law/Act_Business_Collateral_20260204.pdf",
# #   "General Law/CCC_20251111.pdf",


# #   ]

# file_list = {
#    "general_law": {
      
#   # "General Law/Act_Bankruptcy_20260204.pdf": [
#   #     {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
#   #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
#   #     {'key': 'law_name', 'string_value': 'พระราชบัญญัติล้มละลาย พ.ศ. 2483'},
#   #     {'key': 'year', 'numeric_value': 2483 - 543},
#   #     {'key': 'publish_date', 'numeric_value': int(datetime(2483 - 543, 12, 30).timestamp())},

     
#   # ],
#   # "General Law/Act_Business_Collateral_20260202.pdf": [
#   #     {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
#   #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
#   #     {'key': 'law_name', 'string_value': 'พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558'},
#   #     {'key': 'year', 'numeric_value': 2558 - 543},
#   #     {'key': 'publish_date', 'numeric_value': int(datetime(2558 - 543, 10, 31).timestamp())},
     
#   # ],
#   # "General Law/CCC_20251111.pdf": [
#   #     {'key': 'law_level', 'string_value': 'ประมวลกฎหมาย'},
#   #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
#   #     {'key': 'law_name', 'string_value': 'ประมวลกฎหมายแพ่งและพาณิชย์บรรพ 1 และ 2'},
#   #     {'key': 'year', 'numeric_value': 2535 - 543},
#   #     {'key': 'publish_date', 'numeric_value': int(datetime(2535 - 543, 3, 31).timestamp())},
     
#   # ],

#   "General Law/Act_Building_1979.pdf": [
#       {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
#       {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
#       {'key': 'law_name', 'string_value': 'พระราชบัญญัติควบคุมอาคาร พ.ศ. 2522'},
#       {'key': 'year', 'numeric_value': 2522 - 543},
#       {'key': 'publish_date', 'numeric_value': int(datetime(2522 - 543, 5, 8).timestamp())},
     
#   ],

#   },
#   "public_company_law": {
#     "Public Company Law/Act_Public_Company_2535_20260204.pdf": [
#       {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
#       {'key': 'law_type', 'string_value': 'พรบ.บริษัทมหาชนจำกัด กรณีลูกหนี้เป็นบริษัทมหาชน'},
#       {'key': 'law_name', 'string_value': 'พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535'},
#       {'key': 'year', 'numeric_value': 2535 - 543},
#       {'key': 'publish_date', 'numeric_value': int(datetime(2535 - 543, 3, 29).timestamp())},
#     ]
    
#   }
# }

# for file_name, metadata in file_list['general_law'].items():
#   operation = store_file(client, file_name, GENERAL_LAW_FILE_STORE, metadata=metadata)

#   while not operation.done:
#       time.sleep(5)
#       operation = client.operations.get(operation)


# Gemini Prompt Loading

In [ ]:
LEGAL_AI_MODEL = 'gemini-3-flash-preview'
EVAL_AI_MODEL = 'gemini-3-flash-preview'


PROMPT_LEGAL_QA = load_prompt("legal_qa", LEGAL_AI_MODEL, "legal_qa")


In [9]:
def ask_legal_gemini(client, question, file_store_name_list=[GENERAL_LAW_FILE_STORE, PUBLIC_COMPANY_LAW_FILE_STORE, SUPREME_COURT_STATEMENT_FILE_STORE], with_grounding=True, gemini_model=LEGAL_AI_MODEL):
  response = client.models.generate_content(
            model=gemini_model,
            contents=PROMPT_LEGAL_QA.format(question=question),
            config=types.GenerateContentConfig(
                temperature=0.7,
                tools=[
                    types.Tool(
                        file_search=types.FileSearch(
                            file_search_store_names=file_store_name_list
                    
                        )
                    )
                ]
            )
  )

  # Support your response with links to the grounding sources.
  grounding = response.candidates[0].grounding_metadata
  if not grounding:
    ground_txt = 'No grounding sources found'
  else:
    sources = {c.retrieved_context.title for c in grounding.grounding_chunks}
    ground_txt = f'Sources: {sources}'

  if with_grounding:
    try:
      return response, response.text + '\n' + ground_txt 
    except:
      response = client.models.generate_content(
            model=gemini_model,
            contents=PROMPT_LEGAL_QA.format(question=question),
            config=types.GenerateContentConfig(
                temperature=0.7,
                tools=[
                    types.Tool(
                        file_search=types.FileSearch(
                            file_search_store_names=file_store_name_list
                    
                        )
                    )
                ]
            )
      )
  # + '\n' + f'Grounding Score: {grounding.search_entry_point.rendered_content}'

  return response, response.text

In [10]:
question="""
"ข้อ 15. ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเสียชีวิต ธนาคารยังมีบุริมสิทธิในบัญชีเงินฝากหรือไม่ และสามารถดำเนินการอย่างไรได้บ้าง\n"""
response, response_text = ask_legal_gemini(client, question)
print(response_text)

จากการค้นหาข้อกฎหมายและแนวคำพิพากษาศาลฎีกาที่เกี่ยวข้อง ขอสรุปคำตอบกรณีผู้ให้หลักประกัน (เจ้าของบัญชีเงินฝาก) เสียชีวิต โดยมีการจดทะเบียนสัญญาหลักประกันทางธุรกิจ (Business Security Agreement - BSA) ไว้กับธนาคาร ดังนี้

### 1. มาตรากฎหมายที่เกี่ยวข้อง

**พ.ร.บ. หลักประกันทางธุรกิจ พ.ศ. 2558**
*   **มาตรา ๒๙** "ผู้รับหลักประกันมีสิทธิที่จะได้รับชำระหนี้จากทรัพย์สินที่เป็นหลักประกันก่อนเจ้าหนี้สามัญไม่ว่าสิทธิในทรัพย์สินจะโอนไปยังบุคคลภายนอกแล้วหรือไม่"
*   **มาตรา ๔๓** "ในกรณีที่ทรัพย์สินที่เป็นหลักประกันเป็นสิทธิในเงินฝากในสถาบันการเงินและผู้รับหลักประกันเป็นสถาบันการเงินที่รับฝากเงินนั้นไว้เองหรือเป็นผู้รับฝากเงินเพื่อประโยชน์ของผู้รับหลักประกันทั้งหมด ผู้รับหลักประกันอาจนำเงินฝากดังกล่าวหักชำระหนี้ได้ทันทีเมื่อมีเหตุบังคับหลักประกันตามสัญญา แต่ต้องมีหนังสือแจ้งให้ผู้ให้หลักประกันทราบภายในเจ็ดวันนับแต่วันที่ดำเนินการดังกล่าวโดยทางไปรษณีย์ลงทะเบียนตอบรับหรือวิธีการอื่นที่แสดงว่าผู้รับได้รับหนังสือแล้ว..."

**ประมวลกฎหมายแพ่งและพาณิชย์**
*   **มาตรา ๑๖๐๐** "ภายใต้บังคับของบทบัญญัติแห่งปร

# Import Test Cases From File

In [ ]:
# test_df = pd.read_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย (ปรับเพิ่มสำหรับใช้งาน AI)_Internal Use_Extract.xlsx",
#                         sheet_name="Legal Analysis to BU",
#                         skiprows=[0,1,3,4])

# test_df.columns = [x.strip() for x in test_df.columns]
# test_df = test_df.dropna(subset=["ข้อเท็จจริง/คำถาม"])
# test_df.head(3)

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,วันที่ออกความเห็น,Project/Client (กรณีบุคคลธรรมดา ให้ระบุชื่อสกุลบางส่วน เช่น สมxxx โชคxxx),BU,ประเด็นทางกฎหมาย / Key words,ข้อเท็จจริง/คำถาม,สรุปความเห็นทางกฎหมาย,รายละเอียดความเห็นทางกฎหมายแบบที่ส่งให้ BU,กฎหมายที่เกี่ยวข้อง,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น
0,17-Mar-23,NaN,CL-RE,ผู้ให้หลักประกันบัญชีเงินฝากซึ่งจด BSA ไว้เสีย...,ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเส...,ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาค...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1599 และ 160...,NaN,NaN
1,2023-01-26 00:00:00,NaN,CL-RE,สิทธิตาม BSA ในเงินฝากกับบุริมสิทธิเจ้าหนี้ค่า...,ธนาคารต้องส่งมอบเงินฝากซึ่งจด BSA ไว้ ให้แก่เจ...,กรณีที่มีบุริมสิทธิแย้งกับสิทธิตามสัญญาหลักประ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 251, 253(3),...",NaN,NaN
2,2023-02-09 00:00:00,NaN,CL-RE,ข้อยกเว้นที่ผู้จำนองเข้าทำสัญญาค้ำประกันได้,ผู้จำนองที่เป็นผู้ถือหุ้นผู้กู้ เข้าทำสัญญาค้ำ...,ข้อตกลงใดอันมีผลให้ผู้จำนองรับรับผิดอย่างผู้ค้...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 727/1),NaN,NaN


In [ ]:
# test_df['AI_RESPONSE'] = test_df["ข้อเท็จจริง/คำถาม"].apply(lambda x: ask_legal_gemini(client, x, FILE_STORE_NAME)[1])

# test_df.head(5)

KeyboardInterrupt: 

In [ ]:
# test_df.to_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย AI_RESPONSES_2FILES.xlsx")

In [ ]:
# resp = pd.read_excel("/content/drive/MyDrive/KKP/รวบรวมความเห็นและประเด็นทางกฎหมาย AI_RESPONSES_2FILES.xlsx")

# resp.head(3)

,Unnamed: 0,วันที่ออกความเห็น,Project/Client (กรณีบุคคลธรรมดา ให้ระบุชื่อสกุลบางส่วน เช่น สมxxx โชคxxx),BU,ประเด็นทางกฎหมาย / Key words,ข้อเท็จจริง/คำถาม,สรุปความเห็นทางกฎหมาย,รายละเอียดความเห็นทางกฎหมายแบบที่ส่งให้ BU,กฎหมายที่เกี่ยวข้อง,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น,AI_RESPONSE
0,1,2023-04-30 00:00:00,ABCD,CBG,โปรดระบุประเด็นทางกฎหมายของความเห็น เช่น ผู้ให...,โปรดระบุคำถาม/ข้อเท็จจริงที่ได้รับ\nเช่น ผู้ให...,โปรดปรับ/เลือกใช้คำให้อยู่ในรูปแบบธงคำตอบ ที่ส...,โปรดระบุรายละเอียดความเห็นทางกฎหมายแบบที่ส่งให...,โปรดระบุชื่อกฎหมาย (มาตราที่เกี่ยวข้อง),โปรดระบุเลขคำพิพากษาฎีกา (หากมี),ชื่อจริง\n,"คำถาม/ข้อเท็จจริงที่ได้รับคือ: ""ผู้ให้หลักประก..."
1,2,17-Mar-23,NaN,CL-RE,ผู้ให้หลักประกันบัญชีเงินฝากซึ่งจด BSA ไว้เสีย...,ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเส...,ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาค...,NaN,ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 1599 และ 160...,NaN,NaN,ในกรณีที่ผู้ให้หลักประกันนำบัญชีเงินฝากมาจดทะเ...
2,3,2023-01-26 00:00:00,NaN,CL-RE,สิทธิตาม BSA ในเงินฝากกับบุริมสิทธิเจ้าหนี้ค่า...,ธนาคารต้องส่งมอบเงินฝากซึ่งจด BSA ไว้ ให้แก่เจ...,กรณีที่มีบุริมสิทธิแย้งกับสิทธิตามสัญญาหลักประ...,NaN,"ประมวลกฎหมายแพ่งและพาณิชย์ (มาตรา 251, 253(3),...",NaN,NaN,เพื่อตอบคำถามของคุณว่าธนาคารต้องส่งมอบเงินฝากซ...


In [ ]:
# summary_content = ""
# for i, r in resp.iterrows():
#   if i == 0: continue #skip example
#   summary_content += "Q: " + r['ข้อเท็จจริง/คำถาม'] + "\n\n"
#   summary_content += "A: " + r['สรุปความเห็นทางกฎหมาย'] + "\n\n"
#   summary_content += "Gemini: " + r['AI_RESPONSE']  + "\n\n\n"

# print(summary_content)

Q: ผู้ให้หลักประกันนำบัญชีเงินฝากมาจด BSA ต่อมาเสียชีวิต ธนาคารยังมีบุริมสิทธิในบัญชีเงินฝากหรือไม่ และสามารถดำเนินการอย่างไรได้บ้าง
 


A: ความตายของผู้ให้หลักประกันไม่กระทบสิทธิของธนาคารที่มีต่อหลักประกัน เนื่องจากสิทธิในเงินฝากเป็นกองมรดกของผู้ให้หลักประกันและตกทอดไปถึงทายาทของผู้ให้หลักประกันตาม ป.พ.พ. มาตรา 1599 และ 1600 และยังเป็นหลักประกันการชำระหนี้ให้ธนาคารอยู่เนื่องจาก มาตรา 80 พ.ร.บ. หลักประกันทางธุรกิจ พ.ศ. 2558 ไม่ได้กำหนดให้ความตายของผู้ให้หลักประกันเป็นเหตุให้สัญญาหลักประกันทางธุรกิจระงับสิ้นไป

Gemini: ในกรณีที่ผู้ให้หลักประกันนำบัญชีเงินฝากมาจดทะเบียนหลักประกันทางธุรกิจ (BSA) และต่อมาผู้ให้หลักประกันเสียชีวิต ธนาคารในฐานะผู้รับหลักประกันยังมีบุริมสิทธิในบัญชีเงินฝากนั้น และสามารถดำเนินการเพื่อบังคับหลักประกันได้.

**บุริมสิทธิของธนาคาร:**
ตามพระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558 (พ.ร.บ. หลักประกันทางธุรกิจฯ) มาตรา 55 วรรคสอง บัญญัติว่า สิทธิของผู้รับหลักประกันในการบังคับหลักประกันไม่ระงับไปเพราะเหตุที่ผู้ให้หลักประกันถึงแก่ความตาย การที่ผู้ให้หลักประกันเสียชีวิตจึ

In [ ]:
# with open("/content/drive/MyDrive/KKP/AI_RESPONSE_LEGAL_02.txt", 'w', encoding='utf-8') as f:
#   f.write(summary_content)

# Evaluation

In [27]:
from braintrust import Eval
from autoevals import LLMClassifier

import openai
import os

os.environ["BRAINTRUST_API_KEY"] = os.getenv("BRAINTRUST_API_KEY")

In [28]:
legal_reference_scorer = LLMClassifier(
    name="1. Reference",
    model=EVAL_AI_MODEL,
    prompt_template=load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_reference_scorer"),
    choice_scores={
        "อ้างอิงมาตรากฎหมายถูกต้องและครอบคลุมตามเฉลย": 1.0,
        "อ้างอิงมาตรากฎหมายไม่ถูกต้อง หรือไม่เกี่ยวข้อง": 0.0,
        "อ้างอิงมาตรากฎหมายถูกต้อง แต่ไม่ครอบคลุม มีมาตราสำคัญตกหล่น": 0.5
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [29]:
legal_judgement_scorer = LLMClassifier(
    name="2. Judgement",
    model=EVAL_AI_MODEL,
    prompt_template=load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_judgement_scorer"),
    choice_scores={
        "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม": 1.0,
        "ไม่ถูกต้องตามเฉลย": 0.0,
        "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [30]:
legal_suggestion_scorer = LLMClassifier(
    name="3. Conclusion & Suggestion",
    model=EVAL_AI_MODEL,
    prompt_template=load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_suggestion_scorer"),
    choice_scores={
        "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง": 1.0,
        "ไม่ถูกต้องตามเฉลย": 0.0,
        "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [31]:
gemini_distance = LLMClassifier(
    name="Gemini Embedding Similarity",
    model=EVAL_AI_MODEL,
    prompt_template=load_prompt("eval_scorer", EVAL_AI_MODEL, "gemini_distance"),
    choice_scores={
        "Identical Meaning": 1.0,
        "Minor Deviations": 0.8,
        "Major Deviations": 0.4,
        "Unrelated": 0.0
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [32]:
gemini_sim = LLMClassifier(
    name="Gemini Answer Similarity",
    model=EVAL_AI_MODEL,
    prompt_template=load_prompt("eval_scorer", EVAL_AI_MODEL, "gemini_answer_similarity"),
    choice_scores={
        "Equivalent": 1.0,
        "Mostly Similar": 0.7,
        "Partially Similar": 0.3,
        "Different": 0.0
    },
    extra_body={"tool_choice": "auto"},
    max_tokens=None
)

In [33]:
gemini_fact = LLMClassifier(
    name="Gemini Factuality",
    model=EVAL_AI_MODEL,
    prompt_template=load_prompt("eval_scorer", EVAL_AI_MODEL, "gemini_factuality"),
    choice_scores={
        "A": 0.4,
        "B": 0.6,
        "C": 1,
        "D": 0,
        "E": 1
    },
    use_cot=True,
    description="Test whether an output is factual, compared to an original (`expected`) value.",
    extra_body={
        "tool_choice": "none" # Forces Gemini to use the tool call
    },
    max_tokens=None
)



## Load Test Cases

In [34]:
# test_df = pd.read_excel("documents/KKP/test_cases/20260212_Deposit_RAG_Test_Cases.xlsx", 
#                         sheet_name="Legal Analysis to BU",
#                         skiprows=[0, 2])

# test_df = pd.read_excel("documents/KKP/test_cases/20260213_Lending_RAG_Test_Cases.xlsx", 
#                         sheet_name="Sheet1",
# )

test_df = pd.read_excel("documents/KKP/test_cases/20260220_HP_RAG_Test_Cases.xlsx", 
                        sheet_name="เช่าซื้อ",
                        skiprows=[0])

test_df

,ลำดับที่,ประเด็นทางกฎหมาย / Key words,คำถาม,ตัวบทกฎหมาย,คำวินิจฉัย,สรุปและข้อเสนอแนะที่เป็นประโยชน์แก่ธนาคาร,ฎีกาที่เกี่ยวข้อง,ผู้จัดทำความเห็น
0,1,มาตรา 574 ผิดนัด เช่าซื้อ ประกาศ สคบ.,บุคคลธรรมดาเช่าซื้อรถเพื่อใช้ส่วนตัวจากธนาคาร ...,ปพพ. มาตรา 574 ในกรณีผิดนัดไม่ใช้เงินสองคราวต...,การผิดนัดสัญญาเช่าซื้อของบุคคลธรรมดาที่ใช้เพื่...,การผิดนัดสัญญาเช่าซื้อของบุคคลธรรมดาที่ใช้เพื่...,NaN,แจน
1,2,เช่าซื้อ ส่วนลดดอกเบี้ย ประกาศ สคบ.,กรณีบุคคลธรรมดาเช่าซื้อรถเพื่อใช้ส่วนตัวจากธนา...,ประกาศ สคบ 65\nข้อ 5 (9) กรณีที่ผู้เช่าซื้อมีค...,กรณีผู้เช่าซื้อนำเงินมาชำระค่างวดครบก่อนกำหนด ...,กรณีผู้เช่าซื้อนำเงินมาชำระค่างวดครบก่อนกำหนด ...,NaN,มิ้ม\n
2,3,เช่าซื้อ ภาษา ขนาดตัวอักษร ประกาศ สคบ.,ตามประกาศ สคบ. ได้กำหนดให้สัญญาเช่าซื้อรถยนต์ท...,ประกาศ สคบ 65\nข้อ 5 วรรคแรก \nสัญญาเช่าซื้อรถ...,สัญญาเช่าซื้อรถยนต์หรือรถจักรยานยนต์ที่ผู้ประก...,สัญญาเช่าซื้อรถยนต์หรือรถจักรยานยนต์ที่ผู้ประก...,NaN,มิ้ม\n
3,4,เช่าซื้อ อัตราดอกเบี้ย ประกาศ สคบ.,ตามประกาศ สคบ. ให้กำหนดอัตราดอกเบี้ยค่าเช่าซื้...,ประกาศ สคบ 65\nข้อ 5 (1) ข. วรรคสอง \n ในกรณ...,ประกาศ สคบ 65 กำหนดให้สามารถกำหนดดอกเบี้ยในอัต...,ธนาคารต้องกำหนดอัตราดอกเบี้ยให้สอดคล้องกับประก...,NaN,มิ้ม\n
4,5,เช่าซื้อ ดําเนินการจดทะเบียน ประกาศ สคบ.,ตามประกาศ สคบ 65 ผู้ให้เช่าซื้อจะต้องดำเนินการ...,ประกาศ สคบ 65\nข้อ 5 (2)เมื่อผู้เช่าซื้อได้ชํา...,ผู้ให้เช่าซื้อต้องดําเนินการจดทะเบียนรถยนต์ให้...,ผู้ให้เช่าซื้อต้องดําเนินการจดทะเบียนรถยนต์ให้...,NaN,มิ้ม\n
5,6,เช่าซื้อ ค่างวดมาตัดชำระ ประกาศ สคบ.,ตามประกาศ สคบ 65 หากผู้ให้เช่าซื้อประสงค์จะนำเ...,ประกาศ สคบ 65\nข้อ 5 (3) กรณีผู้ให้เช่าซื้อประ...,ผู้ให้เช่าซื้อต้องมีหนังสือแจ้งให้ผู้เช่าซื้อท...,ผู้ให้เช่าซื้อต้องมีหนังสือแจ้งให้ผู้เช่าซื้อท...,NaN,มิ้ม\n
6,7,เช่าซื้อ สิทธิซื้อก่อนประมูล ประกาศ สคบ.,ผู้ให้เช่าซื้อมีหน้าที่จะต้องแจ้งผู้เช่าซื้อ แ...,ประกาศ สคบ 65 ข้อ 5(4) ก.\nข้อ 5 (4) ในกรณีผู้...,ผู้ให้เช่าซื้อต้องมีหนังสือแจ้งให้ผู้เช่าซื้อแ...,ผู้ให้เช่าซื้อต้องมีหนังสือแจ้งให้ผู้เช่าซื้อแ...,NaN,มิ้ม\n
7,8,เช่าซื้อ สิทธิซื้อก่อนประมูล ประกาศ สคบ.,ตามประกาศ สคบ 65 มูลหนี้ส่วนที่ขาดอยู่ตามสัญญา...,ประกาศ สคบ 65 ข้อ 5(4) ก.วรรคสาม\n มูลหนี้ส่วน...,มูลหนี้ส่วนที่ขาดอยู่ตามสัญญาเช่าซื้อให้คำนวณ...,มูลหนี้ส่วนที่ขาดอยู่ตามสัญญาเช่าซื้อให้คำนวณจ...,NaN,มิ้ม\n
8,9,เช่าซื้อ เบี้ยปรับผิดนัดชำระหนี้ ประกาศ สคบ.,ตามประกาศ สคบ 65 ผู้ให้เช่าซื้อสามารถคิดเบี้ยป...,ประกาศ สคบ 65 \nข้อ 6 ข้อสัญญาที่ผู้ประกอบธุรก...,ในกรณีที่ผู้เช่าซื้อผิดนัดชำระค่างวด ผู้ให้เช่...,ในกรณีที่ผู้เช่าซื้อผิดนัดชำระค่างวด ผู้ให้เช่...,NaN,มิ้ม\n
9,10,เช่าซื้อ รถยนต์ถูกทำลาย ไม่ใช่ความผิดของผู้เช่...,ในกรณีที่รถยนต์ที่เช่าซื้อถูกทำลายโดยสิ้นเชิง ...,ประกาศ สคบ 65 \nข้อ 6 ข้อสัญญาที่ผู้ประกอบธุรก...,ในกรณีที่รถยนต์ที่เช่าซื้อถูกทำลายโดยสิ้นเชิง ...,ในกรณีที่รถยนต์ที่เช่าซื้อถูกทำลายโดยสิ้นเชิง ...,NaN,มิ้ม\n


In [35]:
# Prepare data for evaluation
eval_data = []
for i, r in test_df.iterrows():
    # Combine three expected values into one markdown string
    # # Deposit
    # expected_md = f"""## 1. มาตรากฎหมายที่เกี่ยวข้อง
    #                 {r['ตัวบทกฎหมาย']}

    #                 ## 2. คำวินิจฉัย
    #                 {r['คำวินิจฉัย']}

    #                 ## 3. ข้อสรุปและข้อเสนอแนะ
    #                 {r['สรุปและข้อเสนอแนะที่เป็นประโยชน์แก่ธนาคาร']}"""

    # Lending
#     expected_md = f"""## 1. มาตรากฎหมายที่เกี่ยวข้อง
                    # {r['มาตรากฎหมายที่เกี่ยวข้อง']}

                    # ## 2. คำวินิจฉัย
                    # {r['คำวินิจฉัย']}

                    # ## 3. ข้อสรุปและข้อเสนอแนะ
                    # {r['ข้อสรุปและข้อเสนอแนะ']}"""

    
    # HP
    expected_md = f"""## 1. มาตรากฎหมายที่เกี่ยวข้อง
                    {r['ตัวบทกฎหมาย']}

                    ## 2. คำวินิจฉัย
                    {r['คำวินิจฉัย']}

                    ## 3. ข้อสรุปและข้อเสนอแนะ
                    {r['สรุปและข้อเสนอแนะที่เป็นประโยชน์แก่ธนาคาร']}"""
    
    eval_data.append({
        'input': r['คำถาม'],
        'expected': expected_md,
    })
# eval_data.head()

In [36]:
Eval(
    # "KKP Legal Gemini RAG Evaluation - Lending",
    # "KKP Legal Gemini RAG Evaluation - Deposit",
    "KKP Legal Gemini RAG Evaluation - HP",
    data=eval_data,
    experiment_name="Gemini-3.0 Flash Legal V02",
    # experiment_id="63eec50f-04b2-4ab5-8060-730758dfc8e3",
    task=lambda i: ask_legal_gemini(client, i, gemini_model=LEGAL_AI_MODEL)[1],
    scores=[
            gemini_sim,
            legal_reference_scorer,
            legal_judgement_scorer,
            legal_suggestion_scorer],
    # git_metadata_settings={"collect": "none"},
)

Experiment Gemini-3.0 Flash Legal V02 is running at https://www.braintrust.dev/app/KKP/p/KKP%20Legal%20Gemini%20RAG%20Evaluation%20-%20HP/experiments/Gemini-3.0%20Flash%20Legal%20V02


<Task pending name='Task-366' coro=<_EvalCommon.<locals>.run_to_completion() running at /Users/rit/Documents/GitHub/Study RAG/.venv/lib/python3.13/site-packages/braintrust/framework.py:760>>

KKP Legal Gemini RAG Evaluation - HP [experiment_name=Gemini-3.0 Flash Legal V02] (data): 20it [00:00, 30023.65it/s]
KKP Legal Gemini RAG Evaluation - HP [experiment_name=Gemini-3.0 Flash Legal V02] (tasks): 100%|██████████| 20/20 [03:56<00:00, 11.84s/it]



=========================SUMMARY=========================
Gemini-3.0 Flash Legal V02 compared to Gemini-3.0 Pro Legal V02:
15.00% (-) '1. Reference'               score	(1 improvements, 1 regressions)
15.00% (-) '2. Judgement'               score	(3 improvements, 3 regressions)
27.50% (-02.50%) '3. Conclusion & Suggestion' score	(3 improvements, 4 regressions)
14.50% (-) 'Gemini Answer Similarity'   score	(4 improvements, 4 regressions)

4 (-) 'llm_calls'                   	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                  	(0 improvements, 0 regressions)
0 (-) 'errors'                      	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                  	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                 	(0 improvements, 0 regressions)
0tok (-) 'prompt_tokens'               	(0 improvements, 0 regressions)
0tok (-) 'prompt_cached_tokens'        	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'	(0 improvements, 0 regressio